[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-Days-of-ML-Challenge/blob/main/week4_deep_learning/day25_pipelines/day25_notebook.ipynb)

# Day 25 / 42: Pipelines
### #42DaysOfML Challenge

---

## What You'll Learn
- Why writing preprocessing + model as separate steps breaks in production
- How `sklearn.Pipeline` chains steps into one object
- The exact mechanism that causes data leakage without a pipeline
- How `ColumnTransformer` handles mixed-type datasets (numerical + categorical + ordinal)
- How to run `GridSearchCV` directly on a pipeline
- How to save and deploy a pipeline with `joblib`
- A real production bug that pipelines prevent

---

**Datasets:** Breast Cancer Wisconsin (sklearn), synthetic credit risk dataset  
**Prerequisites:** Days 11 (Scaling & Encoding), 18 (Overfitting), 20 (Cross Validation)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_breast_cancer, make_classification

np.random.seed(42)
print('All imports successful.')

## Part 1: The Problem Pipelines Solve

Look at the typical way beginners write ML code:

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
model.fit(X_train_scaled, y_train)
model.predict(X_test_scaled)
```

This looks fine. It has three silent failure modes:

1. **Deployment bug**: the engineer who deploys the model forgets to apply the scaler to new data. Predictions are silently wrong. No error is thrown.
2. **CV leakage**: if you call `scaler.fit_transform(X)` on the full dataset before cross-validation, the scaler has seen the validation fold's statistics. Your CV score is inflated.
3. **Reproducibility**: the scaler and model are separate objects. Saving and loading them must be done separately and in the right order. One wrong step breaks the whole inference chain.

A Pipeline wraps preprocessing and model into **one object**. One `.fit()`. One `.predict()`. One file to save. All three failure modes go away.

## Part 2: Your First Pipeline

The `Pipeline` takes a list of `(name, step)` tuples. Every step except the last must implement `.fit_transform()`. The last step only needs `.fit()` and `.predict()`.

In [ ]:
# Load Breast Cancer dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape}, Test size: {X_test.shape}')
print(f'Feature value ranges (raw):')
print(f'  mean radius: {X[:,0].min():.2f} to {X[:,0].max():.2f}')
print(f'  mean area:   {X[:,3].min():.2f} to {X[:,3].max():.2f}')
print()
print('Features span very different scales. StandardScaler is mandatory before LogReg or KNN.')

In [ ]:
# Build a simple pipeline: StandardScaler -> LogisticRegression
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=10000, random_state=42))
])

# ONE call to fit: StandardScaler.fit_transform(X_train) -> LogisticRegression.fit()
pipe.fit(X_train, y_train)

# ONE call to predict: StandardScaler.transform(X_test) -> LogisticRegression.predict()
y_pred = pipe.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f'Pipeline accuracy: {acc:.4f}')
print()

# Inspect the pipeline
print('Pipeline steps:')
for name, step in pipe.steps:
    print(f'  {name}: {type(step).__name__}')

# Access individual steps by name
print(f'\nScaler mean_ (first 5 features): {pipe.named_steps["scaler"].mean_[:5].round(3)}')
print(f'Classifier coefficients shape:   {pipe.named_steps["clf"].coef_.shape}')

## Part 3: What `.fit()` Actually Does Inside a Pipeline

When you call `pipe.fit(X_train, y_train)`, sklearn does this internally:

```
Step 1: X_scaled = StandardScaler.fit_transform(X_train)  
        → scaler learns mean and std FROM TRAIN DATA ONLY

Step 2: LogisticRegression.fit(X_scaled, y_train)
        → model trains on the scaled train data
```

When you call `pipe.predict(X_test)`, sklearn does:

```
Step 1: X_scaled = StandardScaler.transform(X_test)
        → uses the mean and std learned from TRAIN. Test statistics never touch the scaler.

Step 2: LogisticRegression.predict(X_scaled)
```

The scaler **never** sees the test set. That guarantee is what makes pipelines safe.

## Part 4: Data Leakage — The Bug Pipelines Prevent

Fitting a scaler on all data before cross-validation is one of the most common ML mistakes. It's subtle because it doesn't raise an error and the inflated score looks plausible.

In [ ]:
# Use a small, high-dimensional dataset where the gap is visible
# 100 samples, 50 features, only 5 informative — noisy enough to show leakage
X_small, y_small = make_classification(
    n_samples=100, n_features=50, n_informative=5,
    n_redundant=10, random_state=42
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)
pipe_knn = Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5))])

# === WRONG: fit scaler on the FULL dataset before CV ===
sc_leaky = StandardScaler()
X_leaky = sc_leaky.fit_transform(X_small)   # scaler sees ALL rows including future validation folds
scores_leaky = cross_val_score(knn, X_leaky, y_small, cv=skf)

# === RIGHT: scaler inside pipeline, refitted per fold ===
scores_correct = cross_val_score(pipe_knn, X_small, y_small, cv=skf)

print('=== DATA LEAKAGE DEMONSTRATION ===')
print()
print('WRONG (scaler fit on full dataset before CV):')
print(f'  Fold scores: {scores_leaky.round(3)}')
print(f'  Mean: {scores_leaky.mean():.4f} +/- {scores_leaky.std():.4f}')
print()
print('RIGHT (scaler inside pipeline, refit each fold):')
print(f'  Fold scores: {scores_correct.round(3)}')
print(f'  Mean: {scores_correct.mean():.4f} +/- {scores_correct.std():.4f}')
print()
gap = (scores_leaky.mean() - scores_correct.mean()) * 100
print(f'Reported performance inflated by: {gap:.1f} percentage points')
print()
print('Why it matters: you report a higher number to stakeholders,')
print('ship the model, and real-world performance is lower from day one.')

In [ ]:
# Visualise the gap across folds
fig, ax = plt.subplots(figsize=(9, 4))

folds = [f'Fold {i+1}' for i in range(5)]
x = np.arange(5)
width = 0.35

bars1 = ax.bar(x - width/2, scores_leaky, width, label='Leaky (scaler on all data)',
               color='#E53935', edgecolor='white', alpha=0.85)
bars2 = ax.bar(x + width/2, scores_correct, width, label='Correct (scaler in pipeline)',
               color='#0066CC', edgecolor='white', alpha=0.85)

ax.axhline(scores_leaky.mean(), color='#E53935', linestyle='--', linewidth=1.5,
           label=f'Leaky mean: {scores_leaky.mean():.3f}')
ax.axhline(scores_correct.mean(), color='#0066CC', linestyle='--', linewidth=1.5,
           label=f'Correct mean: {scores_correct.mean():.3f}')

ax.set_xticks(x)
ax.set_xticklabels(folds)
ax.set_ylabel('Accuracy')
ax.set_title('CV Scores: Leaky vs Pipeline (same model, same data)', fontweight='bold')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('The leaky approach consistently reports higher scores — not because the model is better,')
print('but because it already saw the validation data through the scaler.')

## Part 5: ColumnTransformer — Handling Mixed Data Types

Real production datasets rarely have only numerical features. You usually have:
- **Numerical** columns: impute + scale
- **Categorical** (nominal) columns: impute + one-hot encode  
- **Ordinal** columns: encode with the correct order preserved
- **Missing values** in all of the above

`ColumnTransformer` applies different pipelines to different column subsets and concatenates the results. It sits inside the outer Pipeline as one preprocessing step.

In [ ]:
# Create a realistic mixed-type credit risk dataset
np.random.seed(42)
n = 400

df = pd.DataFrame({
    'age':        np.random.randint(22, 65, n).astype(float),
    'income':     np.random.randint(20000, 150000, n).astype(float),
    'loan_amount': np.random.randint(5000, 80000, n).astype(float),
    'city':       np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Hyderabad'], n),
    'risk_level': np.random.choice(['low', 'medium', 'high'], n),
    'default':    np.random.randint(0, 2, n)
})

# Inject realistic missing values
df.loc[np.random.choice(n, 30, replace=False), 'age']    = np.nan
df.loc[np.random.choice(n, 20, replace=False), 'income'] = np.nan
df.loc[np.random.choice(n, 15, replace=False), 'city']   = np.nan

print('Dataset shape:', df.shape)
print()
print(df.head(6).to_string())
print()
print('Missing values per column:')
print(df.isnull().sum().to_string())

In [ ]:
X_df = df.drop('default', axis=1)
y_df = df['default'].values

X_train_df, X_test_df, y_train_df, y_test_df = train_test_split(
    X_df, y_df, test_size=0.2, random_state=42, stratify=y_df
)

# Define column groups
numerical_features = ['age', 'income', 'loan_amount']
categorical_features = ['city']
ordinal_features = ['risk_level']

# Sub-pipeline for numerical columns: impute median -> scale
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Sub-pipeline for categorical columns: impute mode -> one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Sub-pipeline for ordinal columns: encode with explicit order
ordinal_pipeline = Pipeline([
    ('encoder', OrdinalEncoder(categories=[['low', 'medium', 'high']]))
])

# ColumnTransformer applies each sub-pipeline to its column group
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_pipeline,  numerical_features),
    ('cat', categorical_pipeline, categorical_features),
    ('ord', ordinal_pipeline,    ordinal_features)
])

# Full pipeline: preprocessor -> classifier
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# ONE call handles: imputation + scaling + encoding + training
full_pipeline.fit(X_train_df, y_train_df)

y_pred_df = full_pipeline.predict(X_test_df)
acc = accuracy_score(y_test_df, y_pred_df)

print('Full pipeline with ColumnTransformer')
print(f'Test accuracy: {acc:.4f}')
print()
print('What happened under the hood on X_train_df:')
print('  age, income, loan_amount -> median impute -> StandardScaler')
print('  city                     -> mode impute   -> OneHotEncoder (5 cities = 5 columns)')
print('  risk_level               -> OrdinalEncoder (low=0, medium=1, high=2)')
print()

# Check output shape after preprocessing
X_transformed = preprocessor.fit_transform(X_train_df)
print(f'Input shape:  {X_train_df.shape}  (5 columns)')
print(f'Output shape: {X_transformed.shape} (3 numerical + 5 one-hot + 1 ordinal = 9 columns)')

## Part 6: Visualising What Each Step Does

In [ ]:
# Show the transformation at each stage for a single column (income)
income_col = X_train_df['income'].copy()

# After imputation
imputer = SimpleImputer(strategy='median')
income_imputed = imputer.fit_transform(income_col.values.reshape(-1,1)).flatten()

# After scaling
scaler = StandardScaler()
income_scaled = scaler.fit_transform(income_imputed.reshape(-1,1)).flatten()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(income_col.dropna(), bins=25, color='#0066CC', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Raw income\n(NaN count: {income_col.isnull().sum()})', fontweight='bold')
axes[0].set_xlabel('Value')
axes[0].grid(True, alpha=0.3)

axes[1].hist(income_imputed, bins=25, color='#F4A233', edgecolor='white', alpha=0.8)
axes[1].set_title('After median imputation\n(NaN → median value)', fontweight='bold')
axes[1].set_xlabel('Value')
axes[1].grid(True, alpha=0.3)

axes[2].hist(income_scaled, bins=25, color='#00AA44', edgecolor='white', alpha=0.8)
axes[2].set_title('After StandardScaler\n(mean=0, std=1)', fontweight='bold')
axes[2].set_xlabel('Value')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Income column: transformation at each pipeline step', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Raw:     min={income_col.min():.0f}, max={income_col.max():.0f}, NaN={income_col.isnull().sum()}')
print(f'Imputed: min={income_imputed.min():.0f}, max={income_imputed.max():.0f}, NaN=0')
print(f'Scaled:  min={income_scaled.min():.3f}, max={income_scaled.max():.3f}, mean={income_scaled.mean():.5f}')

## Part 7: GridSearchCV on a Pipeline

Because the pipeline is one object, `GridSearchCV` can tune hyperparameters of ANY step inside it. The naming convention is `stepname__parametername` (double underscore).

This means you can tune the imputation strategy, the model depth, and the number of estimators in a single search — and CV is done correctly because the entire pipeline (including preprocessing) is refit from scratch on each fold.

In [ ]:
# GridSearchCV on the full pipeline
# Double underscore __ separates step name from parameter name
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],  # tune imputation
    'clf__n_estimators': [50, 100],                              # tune RF trees
    'clf__max_depth':    [3, 5, None]                            # tune RF depth
}

grid_search = GridSearchCV(
    full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train_df, y_train_df)

print('GridSearchCV on full pipeline')
print(f'Best CV score:  {grid_search.best_score_:.4f}')
print(f'Best params:    {grid_search.best_params_}')
print()

# Evaluate best pipeline on test set
best_pipe = grid_search.best_estimator_
y_pred_best = best_pipe.predict(X_test_df)
print(f'Test accuracy (best pipeline): {accuracy_score(y_test_df, y_pred_best):.4f}')
print()
print('Every combination ran full CV with preprocessing refit from scratch per fold.')
print('No leakage anywhere in the search.')

In [ ]:
# Show all grid search results sorted by score
results_df = pd.DataFrame(grid_search.cv_results_)
cols = ['param_clf__max_depth', 'param_clf__n_estimators',
        'param_preprocessor__num__imputer__strategy',
        'mean_test_score', 'std_test_score', 'rank_test_score']
results_df = results_df[cols].sort_values('rank_test_score')
results_df.columns = ['max_depth', 'n_estimators', 'imputer', 'mean_cv', 'std_cv', 'rank']
results_df['mean_cv'] = results_df['mean_cv'].round(4)
results_df['std_cv']  = results_df['std_cv'].round(4)
print('Top 8 parameter combinations:')
print(results_df.head(8).to_string(index=False))

## Part 8: Saving and Deploying a Pipeline

In production, a model is loaded and called on new incoming data. With a pipeline saved as one object, the deployment code is three lines and preprocessing is guaranteed.

In [ ]:
# Save the best pipeline to disk
joblib.dump(best_pipe, 'credit_risk_pipeline.pkl')
print('Pipeline saved to credit_risk_pipeline.pkl')
print()

# === Simulating what happens in production ===
# A new loan application arrives as raw, unprocessed data
new_applications = pd.DataFrame({
    'age':         [34.0,  np.nan, 52.0],   # one with missing age
    'income':      [65000.0, 42000.0, np.nan],  # one with missing income
    'loan_amount': [25000.0, 15000.0, 60000.0],
    'city':        ['Mumbai', 'Delhi', 'Chennai'],  # Chennai wasn't in training
    'risk_level':  ['medium', 'high', 'low']
})

print('New incoming applications (raw, unprocessed):')
print(new_applications.to_string())
print()

# Production inference: load pipeline, call predict. That's it.
loaded_pipeline = joblib.load('credit_risk_pipeline.pkl')
predictions  = loaded_pipeline.predict(new_applications)
probabilities = loaded_pipeline.predict_proba(new_applications)

print('Predictions on new data:')
for i, (pred, proba) in enumerate(zip(predictions, probabilities)):
    label = 'DEFAULT' if pred == 1 else 'NO DEFAULT'
    print(f'  Application {i+1}: {label} | Probability of default: {proba[1]:.3f}')

print()
print('Pipeline handled missing values, an unseen city (Chennai -> handle_unknown=ignore),')
print('and all encoding automatically. Zero manual preprocessing in production code.')

## Part 9: Real World Problem

### The Deployment Scaler Bug — From the Day 11 Revision Guide

This is documented in the Week 2 Revision Guide. Let's reproduce exactly what happens.

In [ ]:
# === THE BUG: saving model and scaler as separate objects ===

# Training code (written by the data scientist)
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
lr = LogisticRegression(max_iter=10000, random_state=42)
lr.fit(X_train_scaled, y_train)

# DS saves only the model — accidentally forgets the scaler
joblib.dump(lr, 'model_only.pkl')

# Deployment code (written by an engineer, days later)
loaded_lr = joblib.load('model_only.pkl')

# Engineer doesn't know a scaler was applied. Sends raw data.
y_pred_buggy = loaded_lr.predict(X_test)         # X_test is RAW (not scaled)
y_pred_correct = loaded_lr.predict(sc.transform(X_test))  # X_test properly scaled

acc_buggy   = accuracy_score(y_test, y_pred_buggy)
acc_correct = accuracy_score(y_test, y_pred_correct)

print('=== DEPLOYMENT BUG DEMONSTRATION ===')
print()
print(f'Accuracy WITH scaling (correct):   {acc_correct:.4f}')
print(f'Accuracy WITHOUT scaling (bug):    {acc_buggy:.4f}')
print(f'Accuracy lost due to missing scaler: {(acc_correct - acc_buggy)*100:.1f} points')
print()
print('No error was thrown. sklearn does not check whether features are scaled.')
print('The model runs and returns predictions. They are just wrong.')
print()

# === THE FIX: save scaler + model as one pipeline ===
pipe_correct = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=10000, random_state=42))])
pipe_correct.fit(X_train, y_train)
joblib.dump(pipe_correct, 'pipeline_correct.pkl')

# Deployment: one load, one predict. Scaler cannot be forgotten.
loaded_pipe = joblib.load('pipeline_correct.pkl')
acc_pipe = accuracy_score(y_test, loaded_pipe.predict(X_test))

print(f'Pipeline (scaler + model, one object): {acc_pipe:.4f}')
print()
print('The pipeline makes the scaling step structurally impossible to forget.')

## Part 10: Pipeline vs No Pipeline — Full Comparison

In [ ]:
print('=' * 60)
print('PIPELINE VS NO PIPELINE — FULL COMPARISON')
print('=' * 60)
print()

rows = [
    ('Preprocessing on fit',       'Manual, separate calls',      'Automatic, one .fit()'),
    ('Preprocessing on predict',   'Must remember manually',      'Automatic, one .predict()'),
    ('CV data leakage',            'Possible if not careful',     'Impossible by design'),
    ('Saving to disk',             'Scaler + model separately',   'One joblib.dump()'),
    ('Loading in production',      'Load both, apply in order',   'Load one, call .predict()'),
    ('GridSearchCV',               'Tune model params only',      'Tune any step params'),
    ('Reproducibility',            'Manual, error-prone',         'Guaranteed'),
]

header = f'{"Property":<30} {"Without Pipeline":<28} {"With Pipeline"}'
print(header)
print('-' * 85)
for row in rows:
    print(f'{row[0]:<30} {row[1]:<28} {row[2]}')

print()
print('Rule: if preprocessing involves fitting (scalers, imputers, encoders),')
print('it belongs inside a Pipeline. No exceptions.')

## Part 11: Interview Corner

**Q: Your scaler is fit on the full dataset before cross-validation starts. What went wrong and how do you fix it?**

**What they're testing:** Data leakage awareness — one of the most cited production ML mistakes.

**Answer direction:**  
The scaler computes mean and std from the full dataset, including every row that will later appear as validation data in CV. When the model is evaluated on those rows, they have already influenced the feature scaling. The CV score is inflated because the model benefited from statistical information about the validation set during preprocessing. Fix: put the scaler inside a `Pipeline`. When `cross_val_score` runs, it calls `pipeline.fit()` on the training fold — the scaler refits on those rows only — then calls `pipeline.predict()` on the validation fold, where the scaler only transforms using the statistics it learned from the training fold.

---

**Q: In production, when should preprocessing NOT be inside a pipeline?**

**What they're testing:** Whether you know the limits, not just the benefits.

**Answer direction:**  
Two genuine cases: (1) preprocessing is done once on a very large dataset and cached because refitting per fold during CV is too expensive — in this case the preprocessing is done offline and the pipeline starts from the cached features; (2) preprocessing requires external data (e.g. a lookup table that maps raw IDs to embeddings) that isn't part of the sklearn API. In both cases the preprocessing lives outside the pipeline with explicit documentation and validation tests to ensure it is applied correctly at inference time.

## Part 12: ML Spotlight

### scikit-learn Pipeline — The Production Standard

`sklearn.pipeline.Pipeline` was introduced in scikit-learn 0.9 (2012) and has become the standard pattern for production ML in Python. Every major ML platform — Google Vertex AI, AWS SageMaker, Azure ML — expects models as pipeline objects for the same reason: it enforces that preprocessing is always applied, in the right order, with parameters learned only from training data.

From sklearn 1.1 onward, `set_output(transform='pandas')` lets pipelines return DataFrames instead of arrays, preserving column names through the transformation chain. From sklearn 1.2, `ColumnTransformer` became significantly faster with sparse output improvements.

**Read:** https://scikit-learn.org/stable/modules/compose.html

## Part 13: Practice Exercise

Build a pipeline on the Titanic dataset with:
1. Numerical features: `Age`, `Fare` — median impute + StandardScaler
2. Categorical features: `Sex`, `Embarked` — mode impute + OneHotEncoder
3. Classifier: `RandomForestClassifier`

Then:
- Run 5-fold StratifiedKFold CV and report mean accuracy
- Use GridSearchCV to tune `n_estimators` and `max_depth`
- Save the best pipeline with joblib
- Predict on a new passenger: Age=29, Fare=52.0, Sex='female', Embarked='S'

In [ ]:
# Load Titanic data
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)

features = ['Age', 'Fare', 'Sex', 'Embarked']
X_titanic = titanic[features].copy()
y_titanic = titanic['Survived'].values

print('Titanic dataset:', X_titanic.shape)
print('Missing values:')
print(X_titanic.isnull().sum().to_string())
print()

# YOUR CODE HERE
# Hint: define num_features, cat_features
# Build num_pipe, cat_pipe
# Build preprocessor with ColumnTransformer
# Build full_pipe with preprocessor + RandomForestClassifier
# Run cross_val_score with StratifiedKFold
# Run GridSearchCV
# Save and load
# Predict on new passenger

# ====== SOLUTION ======
num_f = ['Age', 'Fare']
cat_f = ['Sex', 'Embarked']

num_p = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_p = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

prep = ColumnTransformer([('num', num_p, num_f), ('cat', cat_p, cat_f)])

titanic_pipe = Pipeline([
    ('prep', prep),
    ('clf',  RandomForestClassifier(n_estimators=100, random_state=42))
])

# CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(titanic_pipe, X_titanic, y_titanic, cv=skf, scoring='accuracy')
print(f'CV scores: {cv_scores.round(4)}')
print(f'Mean: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

# GridSearchCV
param_grid_t = {'clf__n_estimators': [50, 100], 'clf__max_depth': [3, 5, None]}
gs_t = GridSearchCV(titanic_pipe, param_grid_t, cv=5, scoring='accuracy', n_jobs=-1)
gs_t.fit(X_titanic, y_titanic)
print(f'Best params: {gs_t.best_params_}')
print(f'Best CV score: {gs_t.best_score_:.4f}')

# Save and load
joblib.dump(gs_t.best_estimator_, 'titanic_pipeline.pkl')
loaded_titanic = joblib.load('titanic_pipeline.pkl')

# Predict on new passenger
new_passenger = pd.DataFrame({
    'Age': [29.0], 'Fare': [52.0], 'Sex': ['female'], 'Embarked': ['S']
})
pred = loaded_titanic.predict(new_passenger)[0]
proba = loaded_titanic.predict_proba(new_passenger)[0]
print()
print(f'New passenger prediction: {"SURVIVED" if pred == 1 else "DID NOT SURVIVE"}')
print(f'Survival probability: {proba[1]:.3f}')

## Summary

| Concept | Key Point |
|---------|----------|
| Pipeline | Chains steps into one object. One `.fit()`, one `.predict()`, one `.pkl` file. |
| Data leakage in CV | Fitting a scaler on all data before CV inflates reported scores. Pipeline prevents it. |
| ColumnTransformer | Applies different sub-pipelines to different column groups. Concatenates results. |
| GridSearchCV on pipeline | Tune any step's parameters using `stepname__param` double underscore syntax. |
| Deployment | `joblib.dump()` saves the entire pipeline. `joblib.load()` + `.predict()` is the only production code needed. |
| The deployment bug | Saving model without scaler returns wrong predictions silently. Pipeline makes this structurally impossible. |

---

**Tomorrow — Day 26: Neural Networks**  
Layers, weights, activation functions, forward pass, backpropagation. Building one from scratch in NumPy before moving to Keras.

---

**GitHub:** [42-Days-of-ML-Challenge](https://github.com/VaishnaviJagtap18/42-Days-of-ML-Challenge)  
**LinkedIn:** [Vaishnavi Jagtap](https://www.linkedin.com/in/vaishnavi-jagtap18)

#42DaysOfML #MachineLearning #Pipelines #sklearn #Python #MLEngineer